# 订单分批问题

**类别：** 路径规划

来源：[https://www.hexaly.com/templates/order-batching-problem](https://www.hexaly.com/templates/order-batching-problem)


## 问题描述

订单分批问题出现在仓库物流场景中。在该问题中，一组订单（每个订单由位于具有平行通道的矩形仓库中特定位置的商品组成）需要被分组成批次。对于每个批次，一个拣货员在一次仓库巡回中收集所有商品，从仓库起点出发并返回。

一个关键约束是拣货员有限的承载能力：每个批次只能包含一定数量的商品。完成一个批次所行驶的距离由 S 形路径策略确定，即拣货员沿每条通道全长移动，每次往返改变方向（从前到后，然后从后到前）。

总体目标是以最小化拣货员总行驶距离的方式将订单组织成批次。

	

### 建模要点

- 使用 [JSON 模块](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/jsonmodule.html) 读取输入文件
- 使用 [set 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 建模批次
- 使用 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 计算每个批次中的商品数量以及拣货员访问的通道


## 数据

订单分批问题的实例采用 **JSON** 格式。

每个文件包含以下字段：

- “instanceName”：实例的名称

- “nbOrders”：客户订单的总数

- “capacity”：每个批次允许的最大商品数量

- “nbAisles”：仓库中物理通道的数量

- “maxNbBatches”：批次数量的上界

- “aisleTraversal”：每条通道的长度（距离单位）

- “gap”：相邻两条通道中心线之间的距离

- “depotOffset”：从仓库起点到第一条通道的距离

- “orders”：该字段包含一个客户订单列表。每个订单的定义如下：

- “nbItems”：订单中的商品数量
- “aislesToVisit”：为收集该订单需要访问的通道索引

我们依赖标准 JSON 库来读取实例数据并为每个 API 模型写出求解结果：

- **C#**：Newtonsoft.Json
- **Java**：gson 2.8.8
- **Python**：json
- **C++**：nlohmann/json.hpp

对于 **Hexaly Modeler**，使用内置的 [JSON 模块](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/jsonmodule.html)。


## 模型

订单分批问题的 Hexaly 模型[使用 set 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)，每个 set 表示分配给一个批次的订单集合。

首先，这些 set 被约束为构成一个 [partition](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#partition)，即每个订单必须恰好分配到一个批次。

我们使用对 set 的可变参数 **sum** 算子和一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算一个批次中的总商品数量，该函数返回每个订单的商品数量。请注意，求和中的项数在搜索过程中会动态变化，set 的大小也会变化。然后我们可以将商品总数量约束为不超过拣货员的承载能力。

然后，对于每个批次，我们确定需要访问哪些通道。如果批次中至少有一个订单的商品存放在某条通道中，则该通道被访问。基于此，我们使用另一个 lambda 函数计算访问的通道数量以及所访问的最远通道的索引。

然后，按如下方式计算一个批次的 **S 形遍历距离**：拣货员从仓库起点出发，首先水平移动到达最远被访问的通道（距离与 *gap* x *maxVisitedAisle* 成比例），然后遍历每条被访问通道的全长（根据奇偶性调整，以便拣货员始终从前侧离开），最后返回仓库起点（见下文）。在批次为空的特殊情况下，其距离为零。

总体而言，目标是最小化所有批次的总行驶距离。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import json
import sys

def read_instance(input_file):
    with open(input_file) as f:
        data = json.load(f)

    nb_orders = data["nbOrders"]                # Number of orders
    capacity = data["capacity"]                 # Capacity of the picker
    nb_aisles = data["nbAisles"]                # Number of aisles
    max_nb_batches = data["maxNbBatches"]       # Upper bound on the number of batches
    aisle_traversal = data["aisleTraversal"]    # Length of each aisle
    depot_offset = data["depotOffset"]          # Distance from the depot to the first aisle
    gap = data["gap"]            # Distance between the center lines and two adjacent aisles

    nb_items = [0] * nb_orders                                  # Number of items in each order
    visits_aisle = [[0] * nb_orders for _ in range(nb_aisles)]  # Aisles visited by each order
    orders = data["orders"]
    for i in range(nb_orders):
        nb_items[i] = orders[i]["nbItems"]
        for a in orders[i]["aislesToVisit"]:
            visits_aisle[a][i] = 1

    return nb_orders, capacity, nb_aisles, max_nb_batches, aisle_traversal, gap, \
            depot_offset, nb_items, visits_aisle

# Create and solve the model
def main(input_file, output_file, time_limit):
    # Read instance data
    nb_orders, capacity, nb_aisles, max_nb_batches, aisle_traversal, gap, \
            depot_offset, nb_items, visits_aisle = read_instance(input_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        # Declare the optimization model
        model = optimizer.model

        # Set decision: batch[k] contains orders assigned to batch k
        batch = [model.set(nb_orders) for _ in range(max_nb_batches)]

        # Partition constraint: each order is assigned to exactly one batch
        model.constraint(model.partition(batch))

        # Initialize batch properties
        batch_count = [None] * max_nb_batches
        distance = [None] * max_nb_batches
        visited = [[None] * nb_aisles for _ in range(max_nb_batches)]

        # Create arrays for aisle visits
        visits_aisle_array = [model.array(visits_aisle[a]) for a in range(nb_aisles)]

        # Create lambda function for nbItems (reused across batches)
        nb_items_array = model.array(nb_items)
        nb_items_lambda = model.lambda_function(lambda i: nb_items_array[i])

        for k in range(max_nb_batches):
            # Capacity constraint: total number of items of orders assigned to batch k is limited
            batch_items = model.sum(batch[k], nb_items_lambda)
            model.constraint(batch_items <= capacity)

            # Expressions for objective function
            # Batch size
            batch_count[k] = model.count(batch[k])
            # Check which aisles are visited by current batch k
            for a in range(nb_aisles):
                arr = visits_aisle_array[a]
                visited[k][a] = model.sum(batch[k], model.lambda_function(lambda i: arr[i])) >= 1

            # Number of visited aisles
            nb_visited = model.sum(visited[k])
            # Farthest visited aisle
            max_visited = model.max(visited[k][a] * a for a in range(nb_aisles))

            # S-shape traversal distance (full traversal of every visited aisle)
            distance[k] = (batch_count[k] > 0) * (
                    2 * depot_offset
                    + 2 * gap * max_visited
                    + (nb_visited + nb_visited % 2) * aisle_traversal
            )

        # Minimize total distance
        total_distance = model.sum(distance)

        model.minimize(total_distance)
        model.close()

        # Parameterize the solver
        optimizer.param.time_limit = time_limit
        optimizer.solve()

        # Write the solution in a JSON file
        #   - Objective function
        #   - Content of each batch
        #       | Computed orders
        #       | Travelled Distance
        #       | Visited Aisles
        feasible_list = ["HxSolutionStatus.FEASIBLE", "HxSolutionStatus.OPTIMAL"]
        if (str(optimizer.solution.status) in feasible_list):
            if output_file is not None:
                sol = {"objective": int(total_distance.value), "batches": []}
                for k in range(max_nb_batches):
                    c = batch_count[k].value
                    if c > 0:
                        sol["batches"].append({
                            "orders": list(batch[k].value),
                            "distance": int(distance[k].value),
                            "visitedAisles": [
                                a for a in range(nb_aisles)
                                if visited[k][a].value == 1
                            ],
                        })
                with open(output_file, "w") as f:
                    json.dump(sol, f, indent = 2)
                    f.write("\n")
                print("Solution written in file ", output_file)
        else:
            print("No feasible solution have been found within the allowed time.")

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python order_batching_problem.py inputFile [outputFile] [timeLimit]")
        sys.exit(1)

    input_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 20
    main(input_file, output_file, time_limit)
